In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

def load_environment():
    """Load environment variables and return them as a dictionary."""
    load_dotenv(Path("../utils/.env"))  # Loads .env from project root (works if run from notebook too)
    env_vars = {
        "IMAGE_FOLDER": os.getenv("IMAGE_FOLDER"),
        "EXTRACTEDDATASET_FOLDER": os.getenv("EXTRACTEDDATASET_FOLDER"),
        "DATASETS_FOLDER": os.getenv("DATASETS_FOLDER"),
        "ElevationDataset": os.getenv("ElevationDataset"),
        "LandCoverDataset": os.getenv("LandCoverDataset"),
        "GeoBoundaries": os.getenv("GeoBoundaries"),
        "EXTRACTEDELEVATION_FOLDER": os.getenv("EXTRACTEDELEVATION_FOLDER"),
        "EXTRACTEDLANDCOVER_FOLDER": os.getenv("EXTRACTEDLANDCOVER_FOLDER"),
        "EXTRACTEDGEOBOUNDARIES_FOLDER": os.getenv("EXTRACTEDGEOBOUNDARIES_FOLDER")
    }
    return env_vars

folders = load_environment()
image_folder = folders["IMAGE_FOLDER"]
extractedData_folder = folders["EXTRACTEDDATASET_FOLDER"]
datasets_folder = folders["DATASETS_FOLDER"]
landCover_folder = folders["LandCoverDataset"]
elevation_folder = folders["ElevationDataset"]
geoboundaries_folder = folders["GeoBoundaries"]
extracted_elevation_folder = folders["EXTRACTEDELEVATION_FOLDER"]
extracted_landcover_folder = folders["EXTRACTEDLANDCOVER_FOLDER"]
extracted_geo_boundaries_folder = folders["EXTRACTEDGEOBOUNDARIES_FOLDER"]

In [ ]:
# Create a 0.1° grid over the Algerian and Tunisian landcover footprints,
# keep only geometries (no land-cover attributes), and export GeoJSON + centroids CSV.
import numpy as np
from shapely.geometry import box
from shapely.ops import unary_union
import pandas as pd

# Parameters
cell_size = 0.1  # degrees (approximate; assumes EPSG:4326)

# Paths (these names come from earlier cells in this notebook)
dz_path = f"{extracted_landcover_folder}/landcover_algeria.geojson"
tn_path = f"{landCover_folder}/Tunisia/geonetwork_landcover_tun_gc_adg/tun_gc_adg.shp"

# Read input layers (Algeria: pre-extracted geojson; Tunisia: original shapefile)
dz = gpd.read_file(dz_path)
tn = gpd.read_file(tn_path)

# Ensure geographic CRS (degrees) for a simple degree-based grid
dz = dz.to_crs(epsg=4326)
tn = tn.to_crs(epsg=4326)

# Build a single study area union from both layers
study_area = unary_union(list(dz.geometry) + list(tn.geometry))
minx, miny, maxx, maxy = study_area.bounds

# Create grid cell origins (aligned to multiples of cell_size)
start_x = np.floor(minx / cell_size) * cell_size
start_y = np.floor(miny / cell_size) * cell_size
end_x = np.ceil(maxx / cell_size) * cell_size
end_y = np.ceil(maxy / cell_size) * cell_size
xs = np.arange(start_x, end_x, cell_size)
ys = np.arange(start_y, end_y, cell_size)

# Build polygons for the grid
polys = []
for x in xs:
    for y in ys:
        polys.append(box(x, y, x + cell_size, y + cell_size))

grid = gpd.GeoDataFrame({'geometry': polys}, crs='EPSG:4326')

# Clip / intersect grid with study area so cells are limited to the footprints
# Keep any cell that intersects the study area and set geometry to the intersection
intersecting = grid[grid.intersects(study_area)].copy()
intersecting['geometry'] = intersecting.geometry.intersection(study_area)
# Remove empty geometries that might result from topology issues
intersecting = intersecting[~intersecting.is_empty].reset_index(drop=True)

# Drop any attributes: keep only geometry (this yields 'no land cover characteristics')
grid_empty = intersecting[['geometry']].copy()

# Output paths
out_geo = f"{extracted_landcover_folder}/grid_0.1_empty_landcover.geojson"
out_centroids = f"{extracted_landcover_folder}/grid_0.1_empty_landcover_centroids.csv"

# Save GeoJSON (one geometry per feature, no other properties)
grid_empty.to_file(out_geo, driver='GeoJSON')

# Save centroids as a CSV with lon/lat for each cell (useful for sampling/joins)
centroids = grid_empty.geometry.centroid
centroids_df = pd.DataFrame({'lon': centroids.x, 'lat': centroids.y})
centroids_df.to_csv(out_centroids, index=False)

print(f'Created grid cells: {len(grid_empty)}')
print('Saved:', out_geo)
print('Saved:', out_centroids)